In [ ]:
# Imports
import os
from pathlib import Path
import json
import torch
from faster_whisper import WhisperModel
from pydub import AudioSegment
from nemo.collections.asr.models import SortformerEncLabelModel
import numpy as np

# File section and model size
audio_path = Path(r"C:\Users\Somlab\Downloads\audio1365983309.16k.wav")
whisper_size = "small" # tiny / base / small / medium / large-v3 (small fits on my Nvidia 1080 GPU)

# Check if CUDA is installed properly and update settings. Cannot do diarization without CUDA.
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "auto" if device == "cuda" else "int8" # Some systems to float16 or float32, auto lets it do either
print(f"Using device: {device} ({compute_type})")

c:\Users\Somlab\miniforge3\envs\bb_voice_transcription\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-07-17 17:51:34 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
W0717 17:51:34.093000 15348 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Using device: cuda (auto)


### Raw transcription
Transcribe the audio file aithout attempting to identify the speaker. On my computer this took ~15 minutes for a 1 hour interview with CPU only and ~3 minutes with GPU.
Note that GPU requires CUDA 12x.

In [2]:
# Initialize the model (model size, device [cuda or cpu], compute_type).
# word_timestamps=True gives per-word times needed for speaker assignment.
model = WhisperModel(whisper_size, device=device, compute_type=compute_type)
segments, info = model.transcribe(str(audio_path), beam_size=5, word_timestamps=True)

# transcribe() returns a generator; materialize it so we can reuse the segments.
segments = list(segments)
print("Detected language '%s' with probability %f" % (info.language, info.language_probability))
for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

Detected language 'en' with probability 0.998535
[0.00s -> 2.48s]  and perfect.
[3.30s -> 5.14s]  Okay, so for this first section,
[5.34s -> 7.32s]  I'm just gonna be asking about, you know,
[7.42s -> 9.90s]  how life has been since your testing ended
[9.90s -> 11.94s]  and your thoughts and feelings about that.
[12.90s -> 15.60s]  Thinking about your breast sensation now,
[16.30s -> 17.96s]  how do you think it compares
[17.96s -> 20.02s]  to what you felt during testing?
[22.36s -> 25.52s]  I feel as the time is progressing,
[25.96s -> 29.08s]  I feel that I'm getting more sensation,
[29.08s -> 32.64s]  the sensation feels like it's moving
[32.64s -> 34.40s]  more towards the center of my breast.
[34.52s -> 37.32s]  I was pretty much only feeling a lot of the sensation
[37.32s -> 43.08s]  just at the top, almost where nerve endings weren't removed,
[43.34s -> 50.30s]  but now I feel like where I get sensation at not,
[50.44s -> 54.04s]  it's not just at the top section of my chest ca

In [11]:
import numpy as np
max_audio_length = 600
segment_times = np.zeros((len(segments), 3))
for i, segment in enumerate(segments):
    segment_times[i,0] = segment.start
    segment_times[i,1] = segment.end

segment_times[1:,2] = segment_times[1:,1] - segment_times[:-1,0]

# Loop through segments to find gaps and diarize in chunks
start_time, stop_time = -1, max_audio_length
speaker_times = []
while start_time < segment_times[-1,1]:
    # Find the indices of the segments in the range of the start and stop time
    sub_seg_idx = np.where(((segment_times[:,0] > start_time) & 
                            (segment_times[:,1] < stop_time) & 
                            (segment_times[:,2] > 1)))[0]
    start_idx = sub_seg_idx[0]
    stop_idx = sub_seg_idx[-1]
    print(stop_idx)

    # Update 
    start_time = segment_times[stop_idx,1]
    stop_time = start_time + max_audio_length

178
346
516
683
839
1011
1043


array([False,  True,  True, ..., False, False, False], shape=(1044,))